# CHP 08 Extending PySpark with Python: RDD and UDFs

RDD: Resilient Distributed DataSet (underlying object implementing dataframes. Act as a collection of objects instead of a set of rows and columns)
 - can think of each row of a dataframe as an object in an RDD

UDF: User Defined Function

Objects in an RDD are typically modified through the following operations, map(), filter(), and reduce().
These three options are akin to the functional programming concepts map, filter, and reduce found in Java Streams and JavaScript arrays.
Each of these operations are "higher order" functions since they take in other functions as inputs.

- map() -> applies an input function to every element in a collection
- filter() -> picks out/in elements based on an input fiter criterion function
- reduce() -> consolidates a collection of elements into a single element based off of an input consolidation function and an optional default initial value (e.g. for single element lists)

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

sc = spark.sparkContext

In [7]:
collection = [1, "two", 3.0, ("four", 4), {"five":5}]
# create a resilient distributed dataset from the array by parallizing it via the spark context
collection_rdd = sc.parallelize(collection)

In [8]:
print(collection_rdd)

ParallelCollectionRDD[2] at readRDDFromFile at PythonRDD.scala:289


In [13]:
# mapping

from py4j.protocol import Py4JJavaError

def add_one(value):
    """We expect this function to throw an error when it operates on a non-numeric type"""
    return value + 1;

collection_rdd_m = collection_rdd.map(add_one) # attempt to apply the "add_one" to every element in the RDD

try:
    print(collection_rdd_m.collect()) # collect() materializes an RDD into a python lis on the master node
except Py4JJavaError as e:
    pass



25/01/29 07:25:20 ERROR Executor: Exception in task 11.0 in stage 5.0 (TID 71)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/util.py", line 83, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_19041/3692626150.py", line 7, in add_one
TypeError: unsupported operand type(s) for +: 'dict' and 'int'

	at org.apache.spark.api.python.BasePythonRunner$Reade

In [19]:
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def safer_add_one(value):
    """same as add_one but safer. in case of type error it will return the original value while logging the error with stack trace"""
    try:
        return value + 1
    except TypeError as er:
        logger.warning(f"Error encountered: {er}", stack_info=True)
        return value

collection_rdd_m = collection_rdd.map(safer_add_one)

print(collection_rdd_m.collect())

[2, 'two', 4.0, ('four', 4), {'five': 5}]


Error encountered: can only concatenate tuple (not "int") to tuple
Stack (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 218, in <module>
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 193, in manager
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 74, in worker
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/home/hubert/spark-